# Kalman & Bayesian Filters — Walkthrough Notebook
### AI for Robotics

This notebook rebuilds the classic *Kalman and Bayesian Filters in Python* progression so it runs
**top-to-bottom in Google Colab with no extra files**.

The original textbook snippets depend on book-only modules (`book_format`, `code.book_plots`,
`code.kf_internal`, `code.mkf_internal`, `code.nonlinear_plots`) and on an `interactive_plot()`
context manager. None of those exist in Colab, so the original code crashes on import. Here every
helper is defined inline, and the only external dependency is `filterpy` (one `pip install`).

**Road map**
1. Descriptive statistics (mean, median, variance, std)
2. The Gaussian (PDF, CDF, what variance means)
3. Simulating noisy measurements
4. The 1-D Kalman filter (predict + update on Gaussians)
5. Multiplying Gaussians & the Kalman gain
6. Tracking a "dog" in 1-D
7. The multivariate Kalman filter with `filterpy.KalmanFilter`

## 0. Setup

Install `filterpy` and import everything once. Run this cell first.

In [ ]:
!pip install filterpy -q

import math
import numpy as np
import matplotlib.pyplot as plt
from numpy.random import randn
from scipy.stats import norm

np.random.seed(13)
plt.rcParams['figure.figsize'] = (9, 4)
print("numpy", np.__version__, "ready")

## 1. Descriptive statistics

`numpy` gives us mean, median, variance and standard deviation directly.

**Bug fixed from the original:** the textbook snippet used a lowercase list `x` but then referenced
an undefined uppercase `X` (`np.var(X)`, `np.std(X)`), and crammed two `print(...)` calls onto one
line with no separator. Both are fixed below. (Variance = std², which we verify.)

In [ ]:
x = [1.85, 2.0, 1.7, 1.9, 1.6]   # five height measurements (metres)

print("mean   ", np.mean(x))
print("median ", np.median(x))
print("var    ", np.var(x), "meters squared")
print('std  {:.4f}'.format(np.std(x)))
print('var  {:.4f}'.format(np.std(x) ** 2))   # var == std**2

## 2. The Gaussian (Normal) distribution

A Gaussian is a **probability density function** defined by a mean $\mu$ and a variance $\sigma^2$:

$$f(x,\mu,\sigma)=\frac{1}{\sigma\sqrt{2\pi}}\exp\!\left(-\frac{(x-\mu)^2}{2\sigma^2}\right)$$

We define `gaussian(x, mean, var)` ourselves instead of importing it from `filterpy.stats`, and we
write our own `plot_gaussian_pdf` so we don't need the book's `interactive_plot()` helper.

In [ ]:
def gaussian(x, mean, var):
    'Normal distribution density for x given a gaussian with given mean and variance var.'
    return (np.exp((-0.5 * (np.asarray(x) - mean) ** 2) / var)
            / math.sqrt(2 * math.pi * var))

def plot_gaussian_pdf(mean, var, xlim=None, xlabel='', ylabel='pdf', mean_line=False):
    if xlim is None:
        sigma = math.sqrt(var)
        xlim = (mean - 4 * sigma, mean + 4 * sigma)
    xs = np.arange(xlim[0], xlim[1], (xlim[1] - xlim[0]) / 500.)
    plt.plot(xs, gaussian(xs, mean, var))
    if mean_line:
        plt.axvline(mean, ls='--', color='k', alpha=0.6)
    plt.xlabel(xlabel); plt.ylabel(ylabel)

# Student height: mean 1.8 m, std 0.1414 m  ->  var = 0.1414**2
plt.figure()
plot_gaussian_pdf(mean=1.8, var=0.1414 ** 2, xlabel='Student Height', ylabel='pdf')
plt.title('Gaussian: student height'); plt.show()

# Temperature sensor: mean 22 C, var 4
plt.figure()
plot_gaussian_pdf(22, 4, mean_line=True, xlabel='deg C')
plt.title('Gaussian: temperature, mean marked'); plt.show()

### 2.1 Area under the curve (the CDF)

The probability that a value lands in a range is the integral of the PDF over that range — the
**cumulative distribution function**. The book imports `norm_cdf` from `filterpy`; we use
`scipy.stats.norm.cdf`, which is equivalent and always available.

In [ ]:
def norm_cdf(rng, mean, var):
    std = math.sqrt(var)
    return norm.cdf(rng[1], mean, std) - norm.cdf(rng[0], mean, std)

print('Probability of range 21.5 to 22.5 is {:.2f}%'.format(norm_cdf((21.5, 22.5), 22, 4) * 100))
print('Probability of range 23.5 to 24.5 is {:.2f}%'.format(norm_cdf((23.5, 24.5), 22, 4) * 100))

### 2.2 Variance = confidence

A **small** variance is a **narrow, tall** curve: high confidence. A large variance is wide and flat:
low confidence. Same mean (23), three different variances.

In [ ]:
xs = np.arange(15, 30, 0.05)
plt.figure()
plt.plot(xs, gaussian(xs, 23, 0.05), label=r'$\sigma^2$=0.05', c='b')
plt.plot(xs, gaussian(xs, 23, 1),    label=r'$\sigma^2$=1',    ls=':',  c='b')
plt.plot(xs, gaussian(xs, 23, 5),    label=r'$\sigma^2$=5',    ls='--', c='b')
plt.legend(); plt.title('Smaller variance = sharper belief'); plt.show()

## 3. Noisy measurements

Say we believe an object is at 10 m. A single noisy sensor reading is unreliable, but the **average**
of many readings converges on the truth. This is the intuition the Kalman filter formalises.

In [ ]:
# A belief: position 10 m, variance 1
plt.figure()
plot_gaussian_pdf(mean=10, var=1, xlim=(4, 16), xlabel='position (m)')
plt.title('Belief: 10 m with uncertainty'); plt.show()

# 500 noisy readings of a true position of 10 m
xs = range(500)
ys = randn(500) * 1 + 10.
plt.figure()
plt.plot(xs, ys, lw=1)
plt.axhline(10, color='k', ls='--')
plt.title('500 noisy sensor readings'); plt.xlabel('sample'); plt.ylabel('position (m)')
plt.show()
print('Mean of readings is {:.3f}'.format(np.mean(ys)))

## 4. The 1-D Kalman filter

The filter keeps its belief as a Gaussian `(mean, variance)` and runs two steps each cycle.

**Predict** — apply the motion model. Adding two independent Gaussians adds their means **and** their
variances, so motion always *increases* uncertainty:

$$\mu = \mu + \mu_{move}, \qquad \sigma^2 = \sigma^2 + \sigma^2_{move}$$

**Update** — multiply the prior belief by the measurement likelihood (both Gaussians). The product is
a Gaussian that is *narrower* than either input — measuring always *reduces* uncertainty:

$$\mu=\frac{\sigma_1^2\mu_2+\sigma_2^2\mu_1}{\sigma_1^2+\sigma_2^2},\qquad
\sigma^2=\frac{\sigma_1^2\sigma_2^2}{\sigma_1^2+\sigma_2^2}$$

**Bug fixed:** the original `likelihood (z, sensor_var)` was missing the `=` assignment.

In [ ]:
def predict(pos, movement):
    'Prior = prior position (mean, var) + movement (mean, var).'
    return (pos[0] + movement[0], pos[1] + movement[1])

def gaussian_multiply(g1, g2):
    mu1, var1 = g1
    mu2, var2 = g2
    mean = (var1 * mu2 + var2 * mu1) / (var1 + var2)
    variance = (var1 * var2) / (var1 + var2)
    return (mean, variance)

def update(prior, likelihood):
    return gaussian_multiply(likelihood, prior)

### 4.1 Multiplying two Gaussians

Multiplying $\mathcal{N}(10,1)$ by itself yields a Gaussian with the **same mean** but **half the
variance** — combining two independent observations makes us more confident.

In [ ]:
z = (10., 1.)                       # N(10, 1)
product = gaussian_multiply(z, z)

xs = np.arange(5, 15, 0.1)
plt.figure()
plt.plot(xs, [gaussian(v, z[0], z[1]) for v in xs],
         label=r'$\mathcal{N}(10,1)$')
plt.plot(xs, [gaussian(v, product[0], product[1]) for v in xs],
         label=r'$\mathcal{N}(10,1)\times\mathcal{N}(10,1)$', ls='--')
plt.legend(); plt.title('Product of two Gaussians'); plt.show()
print('product (mean, var) =', product)

## 5. Tracking a moving object in 1-D

We simulate a "dog" that walks at roughly constant velocity, with noise in both its motion and the
sensor. The book imports `DogSimulation` and `print_gh` from `code.kf_internal`; we define small,
equivalent versions inline so the notebook is self-contained.

In [ ]:
class DogSimulation:
    'Object moving at ~constant velocity with process and measurement noise.'
    def __init__(self, x0=0, velocity=1, measurement_var=0.0, process_var=0.0):
        self.x = x0
        self.velocity = velocity
        self.meas_std = math.sqrt(measurement_var)
        self.proc_std = math.sqrt(process_var)

    def move(self, dt=1.0):
        dx = self.velocity + randn() * self.proc_std
        self.x += dx * dt

    def sense(self):
        return self.x + randn() * self.meas_std

    def move_and_sense(self):
        self.move()
        return self.sense()

def print_gh(prior, posterior, z):
    print('{:5.2f} {:7.2f}\t\t{:.2f}\t{:5.2f} {:7.2f}'.format(
        prior[0], prior[1], z, posterior[0], posterior[1]))

In [ ]:
np.random.seed(13)
process_var = 1.0           # variance in the dog's movement
sensor_var  = 2.0           # variance in the sensor
x        = (0., 400.)       # initial belief: position 0, big variance 400
velocity = 1.0
dt       = 1.0
process_model = (velocity * dt, process_var)

# simulate the dog and collect 10 measurements
dog = DogSimulation(x[0], process_model[0], sensor_var, process_model[1])
zs  = [dog.move_and_sense() for _ in range(10)]

print('PREDICT\t\t\tUPDATE')
print('  x      var\t\t  z\t  x      var')

xs, predictions = [], []
for z in zs:
    prior      = predict(x, process_model)
    likelihood = (z, sensor_var)
    x          = update(prior, likelihood)
    predictions.append(prior[0])
    xs.append(x[0])
    print_gh(prior, x, z)

print('\nfinal estimate:        {:10.3f}'.format(x[0]))
print('actual final position: {:10.3f}'.format(dog.x))

In [ ]:
# Plot the result of the 1-D filter
plt.figure()
plt.plot(zs, 'r.', label='measurements')
plt.plot(xs, 'b-', label='filter estimate')
plt.plot(predictions, 'g--', alpha=0.6, label='prediction (prior)')
plt.legend(); plt.title('1-D Kalman filter tracking'); plt.xlabel('step'); plt.show()

### 5.1 The Kalman gain

The update can be rewritten with a single scalar, the **Kalman gain** $K$, which decides how much to
trust the new measurement versus the current prediction:

$$y = z - x \quad(\text{residual}),\qquad K=\frac{P}{P+R},\qquad x = x + Ky,\qquad P=(1-K)P$$

- $K \to 1$ when the sensor is trusted (small $R$) → follow measurements.
- $K \to 0$ when the prediction is trusted (small $P$) → ignore noisy measurements.

In [ ]:
def update_gain(prior, measurement):
    x, P = prior          # mean and variance of prior
    z, R = measurement    # mean and variance of measurement
    y = z - x             # residual / innovation
    K = P / (P + R)       # Kalman gain
    x = x + K * y         # posterior mean
    P = (P * R) / (P + R) # posterior variance  == (1-K)P
    return x, P, K

# sanity check: this matches gaussian_multiply
post_x, post_P, K = update_gain((10., 4.), (11., 2.))
print('posterior x={:.3f}  P={:.3f}  K={:.3f}'.format(post_x, post_P, K))
print('gaussian_multiply  ->', gaussian_multiply((10., 4.), (11., 2.)))

## 6. The multivariate Kalman filter

Real robots track several correlated variables at once (here **position** and a hidden **velocity**).
The scalars become vectors and matrices:

| 1-D | Multivariate |
|-----|--------------|
| mean $x$ | state vector $\mathbf{x}$ |
| variance $P$ | covariance matrix $\mathbf{P}$ |
| motion $\times f$ | state transition $\mathbf{F}$ |
| — | measurement function $\mathbf{H}$ |

**Predict:** $\mathbf{x}=\mathbf{Fx}+\mathbf{Bu}$, $\;\mathbf{P}=\mathbf{FPF}^\top+\mathbf{Q}$
**Update:** uses the measurement $\mathbf{z}$, noise $\mathbf{R}$ and $\mathbf{H}$.

**Bugs fixed:** the original had `Q = e` (should be `0`), a stray `xlim(4,16)` call written as a
function instead of a keyword, and used filterpy's `predict`/`update`, which we keep.

In [ ]:
from filterpy.kalman import predict as kf_predict, update as kf_update
from filterpy.common import Q_discrete_white_noise

dt = 0.3
F = np.array([[1., dt],
              [0., 1.]])          # constant-velocity state transition
x = np.array([10.0, 4.5])         # start: position 10, velocity 4.5
P = np.diag([500., 49.])          # large initial uncertainty
Q = 0.0                           # process noise (start with none)

x, P = kf_predict(x, P, F, Q)
print('after one predict:')
print('x =', x)
print('P =\n', P)

### 6.1 The covariance ellipse

A 2-D covariance is drawn as an **ellipse**: position and velocity become correlated after a predict
step (the ellipse tilts), which is exactly the information a Kalman filter exploits.
We define our own ellipse plotter instead of importing `filterpy.stats.plot_covariance_ellipse`.

In [ ]:
from matplotlib.patches import Ellipse

def plot_covariance_ellipse(mean, cov, ax=None, edgecolor='k', ls='solid', nstd=1):
    if ax is None:
        ax = plt.gca()
    vals, vecs = np.linalg.eigh(cov)
    order = vals.argsort()[::-1]
    vals, vecs = vals[order], vecs[:, order]
    theta = np.degrees(np.arctan2(vecs[1, 0], vecs[0, 0]))
    width, height = 2 * nstd * np.sqrt(np.maximum(vals, 0))
    e = Ellipse(xy=mean, width=width, height=height, angle=theta,
                facecolor='none', edgecolor=edgecolor, ls=ls, lw=2)
    ax.add_patch(e)
    ax.scatter(*mean, c=edgecolor, s=15)

dt = 0.3
F = np.array([[1., dt], [0., 1.]])
x = np.array([10.0, 4.5])
P = np.diag([500., 500.])
Q = 0.0

fig, ax = plt.subplots()
plot_covariance_ellipse(x, P, ax, edgecolor='r')           # before predict
x, P = kf_predict(x, P, F, Q)
plot_covariance_ellipse(x, P, ax, edgecolor='k', ls='dashed')  # after predict (tilted)
ax.set_xlabel('position'); ax.set_ylabel('velocity')
ax.set_title('Covariance before (red) and after (dashed) a predict step')
ax.autoscale(); plt.show()

### 6.2 Process-noise matrix Q

A realistic constant-velocity model uses `Q_discrete_white_noise` to build the process-noise matrix
from a single scalar variance.

In [ ]:
Q = Q_discrete_white_noise(dim=2, dt=1., var=2.35)
print(Q)

## 7. The full filter with `filterpy.KalmanFilter`

Now we assemble everything into the textbook's constant-velocity tracker, run it on simulated data,
and compare the estimate to the truth and the raw measurements.

In [ ]:
from filterpy.kalman import KalmanFilter

def pos_vel_filter(x, P, R, Q=0., dt=1.0):
    'A KalmanFilter for a constant-velocity model with state [pos, vel].'
    kf = KalmanFilter(dim_x=2, dim_z=1)
    kf.x = np.array([x[0], x[1]])          # location and velocity
    kf.F = np.array([[1., dt],
                     [0., 1.]])            # state transition matrix
    kf.H = np.array([[1., 0.]])            # we only measure position
    kf.R *= R                              # measurement uncertainty
    if np.isscalar(P):
        kf.P *= P
    else:
        kf.P[:] = P                        # covariance matrix
    if np.isscalar(Q):
        kf.Q = Q_discrete_white_noise(dim=2, dt=dt, var=Q)
    else:
        kf.Q = Q
    return kf

In [ ]:
def compute_dog_data(z_var, process_var, count=1, dt=1.):
    'Simulate true positions xs and noisy measurements zs.'
    x, vel = 0., 1.
    z_std = math.sqrt(z_var)
    p_std = math.sqrt(process_var)
    xs, zs = [], []
    for _ in range(count):
        v = vel + (randn() * p_std * dt)
        x += v * dt
        xs.append(x)
        zs.append(x + randn() * z_std)
    return np.array([xs, zs]).T

def run(x0=(0., 0.), P=500, R=0, Q=0, dt=1.0, data=None, count=0, do_plot=True):
    if data is None:
        data = compute_dog_data(R, Q, count)
    kf = pos_vel_filter(x0, R=R, P=P, Q=Q, dt=dt)

    xs, cov = [], []
    for z in data[:, 1]:
        kf.predict()
        kf.update(z)
        xs.append(kf.x)
        cov.append(kf.P)
    xs, cov = np.array(xs), np.array(cov)

    if do_plot:
        plt.figure()
        plt.plot(data[:, 0], 'g-',  label='true position')
        plt.plot(data[:, 1], 'r.',  label='measurements')
        plt.plot(xs[:, 0],   'b-',  label='KF estimate')
        plt.legend(); plt.title('Constant-velocity Kalman tracker')
        plt.xlabel('step'); plt.ylabel('position'); plt.show()
    return xs, cov

In [ ]:
np.random.seed(13)
dt = 0.1
x0 = np.array([0., 0.])
P  = np.diag([3., 1.])

xs, cov = run(x0=x0, P=P, R=10, Q=0.01, count=50, dt=dt)
print('final estimate (pos, vel):', xs[-1])

### 7.1 The estimate sharpens over time

Plotting the position belief as a Gaussian at successive steps shows the curve getting **taller and
narrower** — uncertainty shrinks as the filter accumulates evidence. This is the whole point of a
Kalman filter: it fuses a motion model with noisy sensors to produce a confident, optimal estimate.

In [ ]:
np.random.seed(3)
Ms, Ps = run(x0=np.array([0., 0.]), count=25, R=10, Q=0.01, P=np.diag([3., 1.]), do_plot=False)

plt.figure()
sel = range(0, len(Ms), 7)
xs_axis = np.arange(-5, 25, 0.1)
for i in sel:
    mean, var = Ms[i, 0], Ps[i, 0, 0]
    plt.plot(xs_axis, gaussian(xs_axis, mean, var), label='step {}'.format(i))
plt.legend(); plt.title('Position belief sharpening over time')
plt.xlabel('position'); plt.ylabel('pdf'); plt.show()

## Summary

| Step | What it does | Effect on uncertainty |
|------|--------------|-----------------------|
| **Predict** | apply motion model ($\mathbf{Fx}$, $\mathbf{FPF}^\top+\mathbf{Q}$) | **increases** |
| **Update** | fuse measurement via Kalman gain $K$ | **decreases** |

The Kalman filter is the optimal recursive estimator for linear systems with Gaussian noise. It
represents belief as a Gaussian, predicts forward with the motion model, then corrects with each
measurement — weighting the two by their relative confidence. Run all cells top to bottom to
reproduce every figure.